# Feature Engineering — Preparación del dataset de modelado (Fase 3 CRISP-DM)

Este notebook transforma los eventos brutos de StatsBomb en un dataset tabular con una fila por **(partido, minuto)**, listo para entrenar el modelo de predicción de probabilidad de victoria.

**Estrategia de construcción:**
- Para cada minuto M ∈ [1, 90], se acumulan todos los eventos con `minute ≤ M` y `period ≤ 2`.
- El minuto 90 absorbe el tiempo de descuento (goles en min 91–97) porque es parte del tiempo reglamentario.
- Las prórrogas y tandas de penaltis (periods 3–5) se excluyen de las features pero su impacto queda capturado en `final_result`.

**Output:** `../datasets/match_minute_features.csv` — ~311 760 filas × ~40 columnas.

In [1]:
import pandas as pd
import numpy as np

CSV_PATH = "../datasets/statsbomb_events.csv.gz"
OUT_PATH = "../datasets/match_minute_features.csv"
WINDOW   = 15   # rolling window width in minutes

## 1. Carga y filtrado a tiempo reglamentario

In [2]:
df = pd.read_csv(CSV_PATH, low_memory=False)

# Keep only periods 1 and 2 (regulation). Periods 3-5 = extra time / penalties.
df_reg = df[df["period"] <= 2].copy()

# xg is NaN for every non-shot event (structural). Fill with 0.
df_reg["xg"] = df_reg["xg"].fillna(0.0)

n_extra_matches = df[df["period"] >= 3]["match_id"].nunique()
print(f"Total events:              {len(df):>12,}")
print(f"Regulation events (p1+p2): {len(df_reg):>12,}")
print(f"Matches:                   {df_reg['match_id'].nunique():>12,}")
print(f"Matches with extra time:   {n_extra_matches:>12,} (kept — target captures ET result)")

Total events:                12,188,949
Regulation events (p1+p2):   12,134,916
Matches:                          3,464
Matches with extra time:             55 (kept — target captures ET result)


## 2. Indicadores binarios por evento

Cada evento se convierte en uno o varios indicadores enteros (0/1). El acumulado de estos indicadores a lo largo del partido construirá las features del modelo.

In [3]:
SOT_OUTCOMES = {"Goal", "Saved", "Saved Off Target", "Saved to Post"}
DUEL_WON     = {"Won", "Success In Play", "Success Out"}
RED_TYPES    = {"Red Card", "Second Yellow"}

# StatsBomb pitch: 120 x 80 yards. Attacking third starts at x > 80; penalty box at x > 102.
loc_x = df_reg["loc_x"].fillna(0)

indicators = {
    "goal":           (df_reg["event_type"] == "Shot") & (df_reg["shot_outcome"] == "Goal"),
    "shot":            df_reg["event_type"] == "Shot",
    "shot_on_target": (df_reg["event_type"] == "Shot") & df_reg["shot_outcome"].isin(SOT_OUTCOMES),
    "shot_in_box":    (df_reg["event_type"] == "Shot") & (loc_x > 102),
    "pass":            df_reg["event_type"] == "Pass",
    "complete_pass":  (df_reg["event_type"] == "Pass") & (df_reg["pass_outcome"] == "Complete"),
    "pressure":        df_reg["event_type"] == "Pressure",
    "duel_won":       (df_reg["event_type"] == "Duel") & df_reg["duel_outcome"].isin(DUEL_WON),
    "clearance":       df_reg["event_type"] == "Clearance",
    "block":           df_reg["event_type"] == "Block",
    "carry":           df_reg["event_type"] == "Carry",
    "yellow_card":     df_reg["card_type"] == "Yellow Card",
    "red_card":        df_reg["card_type"].isin(RED_TYPES),
    "attack_third":    loc_x > 80,
}

for name, mask in indicators.items():
    df_reg[name] = mask.astype(np.int8)

FEAT_COLS = list(indicators.keys()) + ["xg"]

print(f"Indicator columns created: {len(FEAT_COLS)}")
df_reg[FEAT_COLS].sum().rename("total").to_frame()

Indicator columns created: 15


,total
goal,9.518000e+03
shot,8.718900e+04
shot_on_target,3.075600e+04
shot_in_box,5.233700e+04
pass,3.373012e+06
complete_pass,2.621936e+06
pressure,1.109311e+06
duel_won,8.183100e+04
clearance,1.581510e+05
block,1.317210e+05


## 3. Sumas acumuladas por (partido, equipo)

Ordenamos los eventos por tiempo y aplicamos `cumsum` dentro de cada grupo `(match_id, is_home)`. El resultado es el estado acumulado justo **después** de cada evento.

In [4]:
df_reg = df_reg.sort_values(["match_id", "is_home", "minute", "second"])

for col in FEAT_COLS:
    df_reg[f"cumul_{col}"] = df_reg.groupby(["match_id", "is_home"])[col].cumsum()

CUMUL_COLS = [f"cumul_{c}" for c in FEAT_COLS]
print(f"Cumulative columns added: {len(CUMUL_COLS)}")
df_reg[["match_id", "is_home", "minute", "second"] + CUMUL_COLS[:5]].head(8)

Cumulative columns added: 15


,match_id,is_home,minute,second,cumul_goal,cumul_shot,cumul_shot_on_target,cumul_shot_in_box,cumul_pass
2296783,7298,False,0,0,0,0,0,0,0
2296785,7298,False,0,0,0,0,0,0,0
2296786,7298,False,0,0,0,0,0,0,1
2296787,7298,False,0,0,0,0,0,0,1
2296788,7298,False,0,0,0,0,0,0,1
2296790,7298,False,0,0,0,0,0,0,2
2296791,7298,False,0,2,0,0,0,0,2
2296792,7298,False,0,2,0,0,0,0,2


## 4. Rejilla de minutos (1–90) y relleno hacia adelante

Queremos exactamente **una fila por minuto por equipo por partido**. El proceso es:
1. Clipar el minuto a 90 → los goles en tiempo de descuento (min 91-97, period 2) quedan asignados al snapshot del minuto 90.
2. Para cada `(match_id, is_home, minute)`, tomar el **último** valor acumulado del minuto.
3. Crear la rejilla completa de minutos 1–90 y hacer *forward fill* para los minutos sin eventos.

In [5]:
# Stoppage time events (period=2, minute > 90) -> absorbed into minute 90 snapshot
df_reg["minute_key"] = df_reg["minute"].clip(upper=90)

minute_state = (
    df_reg
    .groupby(["match_id", "is_home", "minute_key"])[CUMUL_COLS]
    .last()
    .reset_index()
    .rename(columns={"minute_key": "minute"})
)

# Full minute grid: every match x both teams x every minute 1-90
all_match_ids = df_reg["match_id"].unique()
grid = pd.MultiIndex.from_product(
    [all_match_ids, [True, False], range(1, 91)],
    names=["match_id", "is_home", "minute"],
).to_frame(index=False)

minute_full = (
    grid
    .merge(minute_state, on=["match_id", "is_home", "minute"], how="left")
    .sort_values(["match_id", "is_home", "minute"])
)

# Forward-fill within each (match, team): a 0-0 at minute 10 stays 0-0 until a goal
minute_full[CUMUL_COLS] = (
    minute_full
    .groupby(["match_id", "is_home"])[CUMUL_COLS]
    .ffill()
    .fillna(0)   # minutes before the first event of the match
)

expected = len(all_match_ids) * 2 * 90
print(f"Grid shape: {minute_full.shape}")
print(f"Expected:   ({expected:,}, {minute_full.shape[1]})")

Grid shape: (623520, 18)
Expected:   (623,520, 18)


## 5. Features de ventana deslizante (últimos 15 minutos)

El **momentum** reciente es independiente del estado acumulado total. Se calcula eficientemente como la diferencia entre el valor acumulado en el minuto actual y el de 15 minutos atrás (usando `shift(15)` dentro de cada grupo).

Se crean ventanas para xG, tiros y presiones — las features con mayor correlación con el resultado según el EDA.

In [6]:
ROLLING_SRC = {
    "cumul_xg":       f"xg_last{WINDOW}",
    "cumul_shot":     f"shots_last{WINDOW}",
    "cumul_pressure": f"pressures_last{WINDOW}",
}

for cumul_col, roll_name in ROLLING_SRC.items():
    lag = (
        minute_full
        .groupby(["match_id", "is_home"])[cumul_col]
        .shift(WINDOW)
        .fillna(0)
    )
    minute_full[roll_name] = minute_full[cumul_col] - lag

ROLL_COLS = list(ROLLING_SRC.values())
print(f"Rolling features ({WINDOW}-min window):")
minute_full[ROLL_COLS].describe().loc[["min", "mean", "max"]].round(3)

Rolling features (15-min window):


,xg_last15,shots_last15,pressures_last15
min,-0.000,0.000,0.000
mean,0.192,1.857,24.196
max,3.274,16.000,114.000


## 6. Pivotado local / visitante

Actualmente cada fila es `(match_id, is_home, minute)`. Separamos local y visitante y los juntamos en una sola fila por `(match_id, minute)` con sufijos `_home` / `_away`.

In [7]:
RENAME = {
    "cumul_goal":           "goals",
    "cumul_shot":           "shots",
    "cumul_shot_on_target": "shots_on_target",
    "cumul_shot_in_box":    "shots_in_box",
    "cumul_xg":             "xg",
    "cumul_pass":           "passes",
    "cumul_complete_pass":  "complete_passes",
    "cumul_pressure":       "pressures",
    "cumul_duel_won":       "duels_won",
    "cumul_clearance":      "clearances",
    "cumul_block":          "blocks",
    "cumul_carry":          "carries",
    "cumul_yellow_card":    "yellow_cards",
    "cumul_red_card":       "red_cards",
    "cumul_attack_third":   "attacks_third",
    f"xg_last{WINDOW}":        f"xg_last{WINDOW}",
    f"shots_last{WINDOW}":     f"shots_last{WINDOW}",
    f"pressures_last{WINDOW}": f"pressures_last{WINDOW}",
}

minute_full = minute_full.rename(columns=RENAME)
VALUE_COLS  = list(RENAME.values())

home = (
    minute_full[minute_full["is_home"]]
    .drop(columns="is_home")
    .rename(columns={c: f"{c}_home" for c in VALUE_COLS})
)
away = (
    minute_full[~minute_full["is_home"]]
    .drop(columns="is_home")
    .rename(columns={c: f"{c}_away" for c in VALUE_COLS})
)

features = home.merge(away, on=["match_id", "minute"])
print(f"Shape after pivot: {features.shape}")
features.head(3)

Shape after pivot: (311760, 38)


,match_id,minute,goals_home,shots_home,shots_on_target_home,shots_in_box_home,passes_home,complete_passes_home,pressures_home,duels_won_home,...,clearances_away,blocks_away,carries_away,yellow_cards_away,red_cards_away,attacks_third_away,xg_away,xg_last15_away,shots_last15_away,pressures_last15_away
0,7298,1,0.0,0.0,0.0,0.0,7.0,5.0,4.0,0.0,...,0.0,2.0,7.0,0.0,0.0,13.0,0.018856,0.018856,1.0,7.0
1,7298,2,0.0,0.0,0.0,0.0,10.0,7.0,7.0,0.0,...,0.0,2.0,10.0,0.0,0.0,13.0,0.018856,0.018856,1.0,9.0
2,7298,3,0.0,0.0,0.0,0.0,20.0,16.0,11.0,1.0,...,0.0,2.0,16.0,0.0,0.0,25.0,0.018856,0.018856,1.0,16.0


## 7. Features derivadas

Se calculan diferencias, ratios y variables de contexto temporal a partir de las columnas acumuladas.

In [8]:
# Scoreline differentials
features["score_diff"] = features["goals_home"] - features["goals_away"]
features["xg_diff"]    = features["xg_home"]    - features["xg_away"]
features["shots_diff"] = features["shots_home"] - features["shots_away"]

# Possession proxy: share of carries (ball-in-motion events)
total_carries = features["carries_home"] + features["carries_away"]
features["possession_home"] = (
    features["carries_home"] / total_carries.replace(0, np.nan)
).fillna(0.5)   # 0.5 when no carries yet (kick-off minute)

# Pass completion rates
features["pass_completion_home"] = (
    features["complete_passes_home"] / features["passes_home"].replace(0, np.nan)
).fillna(0.0)
features["pass_completion_away"] = (
    features["complete_passes_away"] / features["passes_away"].replace(0, np.nan)
).fillna(0.0)

# Temporal context
features["minutes_remaining"] = 90 - features["minute"]
features["period"]  = (features["minute"] > 45).astype(np.int8) + 1
features["score_tied"] = (features["score_diff"] == 0).astype(np.int8)

print("Derived features added.")
features[["score_diff", "xg_diff", "possession_home",
           "pass_completion_home", "minutes_remaining", "period", "score_tied"]].describe().round(3)

Derived features added.


,score_diff,xg_diff,possession_home,pass_completion_home,minutes_remaining,period,score_tied
count,311760.000,311760.000,311760.000,311760.000,311760.000,311760.0,311760.000
mean,0.158,0.127,0.520,0.765,44.500,1.5,0.469
std,1.304,0.836,0.162,0.092,25.979,0.5,0.499
min,-8.000,-5.124,0.000,0.000,0.000,1.0,0.000
25%,0.000,-0.236,0.401,0.710,22.000,1.0,0.000
50%,0.000,0.047,0.525,0.776,44.500,1.5,0.000
75%,1.000,0.481,0.640,0.832,67.000,2.0,1.000
max,13.000,6.055,1.000,1.000,89.000,2.0,1.000


## 8. Metadatos y variable objetivo

`final_result` (la variable objetivo) está disponible en el dataset de eventos. Se une por `match_id` junto con los metadatos de la competición.

In [9]:
WOMENS_KW = ["Women", "Female", "NWSL", "WSL"]

meta = (
    df.drop_duplicates("match_id")
    .set_index("match_id")[["competition_name", "season_name", "home_team", "away_team", "final_result"]]
)

features = features.join(meta, on="match_id")
features["is_womens"] = (
    features["competition_name"].str.contains("|".join(WOMENS_KW), case=False)
).astype(np.int8)

print("Competitions in dataset:")
features.drop_duplicates("match_id")["competition_name"].value_counts().to_frame("matches")

Competitions in dataset:


,matches
competition_name,
La Liga,868
Ligue 1,435
Premier League,418
Serie A,381
1. Bundesliga,340
FA Women's Super League,326
FIFA World Cup,147
Women's World Cup,116
Indian Super league,115


## 9. Validación del dataset

Tres comprobaciones antes de guardar: dimensiones, ausencia de NaN en columnas clave, y verificación del marcador acumulado en un partido de muestra.

In [10]:
# --- Check 1: Shape ---
n_matches  = features["match_id"].nunique()
expected   = n_matches * 90
print(f"Shape:    {features.shape}")
print(f"Expected: ({expected:,}, ...)  [{n_matches} matches × 90 min]")
assert len(features) == expected, "Row count mismatch!"

Shape:    (311760, 53)
Expected: (311,760, ...)  [3464 matches × 90 min]


In [11]:
# --- Check 2: No NaN in key modelling columns ---
key_cols = [
    "goals_home", "goals_away", "xg_home", "xg_away",
    "score_diff", "possession_home", "minutes_remaining",
    "final_result",
]
nan_counts = features[key_cols].isnull().sum()
print("NaN counts in key columns:")
print(nan_counts.to_frame("NaNs"))
assert nan_counts.sum() == 0, "Unexpected NaN values found!"

NaN counts in key columns:
                   NaNs
goals_home            0
goals_away            0
xg_home               0
xg_away               0
score_diff            0
possession_home       0
minutes_remaining     0
final_result          0


In [12]:
# --- Check 3: Validate running score against raw goals for a sample match ---
# Pick the match with the most goals for a visible test
goals_raw = df[(df["event_type"] == "Shot") & (df["shot_outcome"] == "Goal") & (df["period"] <= 2)]
sample_id = goals_raw.groupby("match_id").size().idxmax()

raw_info   = df[df["match_id"] == sample_id].iloc[0]
feat_match = features[features["match_id"] == sample_id]

# Final scoreline from raw data vs derived goals
raw_goals_home = ((goals_raw["match_id"] == sample_id) & (goals_raw["is_home"])).sum()
raw_goals_away = ((goals_raw["match_id"] == sample_id) & (~goals_raw["is_home"])).sum()
derived_home   = int(feat_match["goals_home"].iloc[-1])
derived_away   = int(feat_match["goals_away"].iloc[-1])

print(f"Match: {raw_info['home_team']} {raw_info['final_score_home']}-{raw_info['final_score_away']} {raw_info['away_team']}")
print(f"Raw regulation goals   — home: {raw_goals_home}, away: {raw_goals_away}")
print(f"Derived (minute 90)    — home: {derived_home},  away: {derived_away}")

print("\nRunning score (every 10 minutes):")
feat_match[feat_match["minute"] % 10 == 0][["minute", "goals_home", "goals_away", "score_diff", "xg_home", "xg_away"]]

Match: United States Women's 13-0 Thailand Women's
Raw regulation goals   — home: 13, away: 0
Derived (minute 90)    — home: 13,  away: 0

Running score (every 10 minutes):


,minute,goals_home,goals_away,score_diff,xg_home,xg_away
26829,10,0.0,0.0,0.0,0.394235,0.00000
26839,20,2.0,0.0,2.0,0.717110,0.00000
26849,30,2.0,0.0,2.0,0.916021,0.01120
26859,40,3.0,0.0,3.0,1.275392,0.01120
26869,50,4.0,0.0,4.0,1.864517,0.01120
26879,60,7.0,0.0,7.0,3.365818,0.01120
26889,70,7.0,0.0,7.0,3.872095,0.01120
26899,80,10.0,0.0,10.0,4.230263,0.02031
26909,90,13.0,0.0,13.0,5.953466,0.02031


In [13]:
# --- Check 4: Target class balance (match level) ---
match_level = features.drop_duplicates("match_id")["final_result"].value_counts()
balance = match_level.to_frame("matches")
balance["pct"] = (balance["matches"] / balance["matches"].sum() * 100).round(1)
print("Target distribution (match level):")
balance

Target distribution (match level):


,matches,pct
final_result,,
home_win,1565,45.2
away_win,1102,31.8
draw,797,23.0


## 10. Guardado

Ordenamos por `(match_id, minute)` para que las filas de cada partido sean contiguas, lo que facilita la división train/test por partido y el análisis posterior.

In [14]:
features = features.sort_values(["match_id", "minute"]).reset_index(drop=True)
features.to_csv(OUT_PATH, index=False)

print(f"Saved: {features.shape[0]:,} rows × {features.shape[1]} columns")
print(f"Path:  {OUT_PATH}")
print("\nColumn overview:")
features.dtypes.value_counts().rename("columns").to_frame()

Saved: 311,760 rows × 53 columns
Path:  ../datasets/match_minute_features.csv

Column overview:


,columns
float64,42
str,5
int64,3
int8,3


In [15]:
# Final column list for reference
model_cols = [c for c in features.columns if c not in
              ["match_id", "competition_name", "season_name", "home_team", "away_team", "final_result"]]
print(f"Model feature columns ({len(model_cols)}):")
print(model_cols)

Model feature columns (47):
['minute', 'goals_home', 'shots_home', 'shots_on_target_home', 'shots_in_box_home', 'passes_home', 'complete_passes_home', 'pressures_home', 'duels_won_home', 'clearances_home', 'blocks_home', 'carries_home', 'yellow_cards_home', 'red_cards_home', 'attacks_third_home', 'xg_home', 'xg_last15_home', 'shots_last15_home', 'pressures_last15_home', 'goals_away', 'shots_away', 'shots_on_target_away', 'shots_in_box_away', 'passes_away', 'complete_passes_away', 'pressures_away', 'duels_won_away', 'clearances_away', 'blocks_away', 'carries_away', 'yellow_cards_away', 'red_cards_away', 'attacks_third_away', 'xg_away', 'xg_last15_away', 'shots_last15_away', 'pressures_last15_away', 'score_diff', 'xg_diff', 'shots_diff', 'possession_home', 'pass_completion_home', 'pass_completion_away', 'minutes_remaining', 'period', 'score_tied', 'is_womens']
